In [13]:
# Imports
import ROOT
import array

from tools import scale_out as so

from emm import data
from emm import fitting
from emm import models
from emm import bias

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Dijet

In [14]:
# Load the data
h_obs, h_fit = data.get_dijet_data()
n_obs = h_obs.Integral()

# Get binning from data
xaxis = h_obs.GetXaxis()
boundaries = [xaxis.GetBinLowEdge(i) for i in range(1, xaxis.GetNbins() + 2)]
binning = ROOT.RooBinning(len(boundaries) - 1, array.array('d', boundaries))

x = ROOT.RooRealVar("x", "Dijet Mass [GeV]", boundaries[0], boundaries[-1])
x.setBinning(binning)

dijet_data = ROOT.RooDataHist("ATLAS_dijet_data", "ATLAS dijet data", ROOT.RooArgList(x), h_obs)
print(f"Data mean: {dijet_data.mean(x)}")

# SW -- Dijet Paper Fit
dh_fit = ROOT.RooDataHist("dh_fit", "dh_fit", ROOT.RooArgList(x), h_fit)
pdf_fit = ROOT.RooHistPdf("pdf_fit", "pdf_fit", ROOT.RooArgSet(x), dh_fit)

fit_options = [
    ROOT.RooFit.IntegrateBins(0.001),
    ROOT.RooFit.PrintLevel(-1),
    # ROOT.RooFit.Offset(True),
    ROOT.RooFit.Strategy(2),
    ROOT.RooFit.Save(),
    # ROOT.RooFit.Range("fit_range")
]

Data mean: 1347.5082817871998


Warning in <TROOT::Append>: Replacing existing TH1: h_observed (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_fit (Potential memory leak).


In [18]:
# Configuration
x_min, x_max = 1100, 10000
x_grid = np.linspace(x_min, x_max, 1000)

# Default
n_seeds = 5
n_toys_per_seed = 10

# Generate seeds
default_seed = 42
np.random.seed(default_seed)
seeds = np.random.randint(0, 10000, size=n_seeds)

In [15]:
# Set up models
toy_models = [
    models.Dijet(x)
]

model_primitives = [
    models.Dijet
]
for k in [3, 4, 5]:
    model_primitives.append(
        models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k,
            data_mean=1347,
            name=f"ExponentialMixture-{k}",
        )
    )

print(f"Will be submitting {n_seeds * len(toy_models)} tasks, with {n_toys_per_seed} toys fitting {len(model_primitives)} models.")

# Fit toy models to data
for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(dijet_data)

Will be submitting 5 tasks, with 10 toys fitting 4 models.
Fitting toy model: Dijet
[#1] INFO:NumericIntegration -- RooRealIntegral::init(Dijet_Int[x]) using numeric integrator RooRombergIntegrator to calculate Int(x)
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
Minuit2Minimizer: Minimize with max-calls 1500 convergence for edm < 1 strategy 1
Minuit2Minimizer : Valid minimum - status = 1
FVAL  = 186122585.496872425
Edm   = 0.000173007186865297493
Nfcn  = 210
p1	  = 4.26419	 +/-  0.0391819	(limited)
p2	  = -7.81763	 +/-  0.0153686	(limited)
p3	  = -0.482621	 +/-  0.00272103	(limited)
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization


Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =       189552463.5 Edm =        5213336.87 NCalls =     19
Info in <Minuit2>: MnSeedGenerator Initial state  
  Minimum value : 189552463.5
  Edm           : 5213336.87
  Internal parameters:	[    -0.2013579208    -0.5235987756    -0.1001674212]	
  Internal gradient  :	[     -19626616.26      151290737.4     -793442412.9]	
  Internal covariance matrix:
[[  1.4001507e-08              0              0]
 [              0  3.1607782e-10              0]
 [              0              0  1.3065251e-11]]]
Info in <Minuit2>: VariableMetricBuilder Start iterating until Edm is < 0.001 with call limit = 1500
Info in <Minuit2>: VariableMetricBuilder    0 - FCN =       189552463.5 Edm =        5213336.87 NCalls =     19
Info in <Minuit2>: VariableMetricBuilder    1 - FCN =       187605714.1 Edm =       70358.84936 NCalls =     31
Info in <Minuit2>: VariableMetr

In [ ]:
# Run jobs
tasks = []
for seed in seeds:
    for toy_model in toy_models:
        tasks.append(
            so.Task(
                bias.run_bias_fits,
                x, toy_model, model_primitives,
                seed, n_toys_per_seed,
                n_obs, x_grid
            )
        )

# _ = so.run_tasks(
#     tasks,
#     use_condor=True,
#     condor_job_name="bias",
#     env_wrapper=so.run_in_mamba,
#     clear_logs=True
# )

# Diphoton

In [ ]:
# # Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = emm.data.get_diphoton_data(sort_and_index=True, tree=True)
data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = data.numEntries()
data_mean = data.mean(x)
print(f"Data mean: {data_mean}")

In [ ]:
# Configuration
x_min, x_max = 500, 3500
x_grid = np.linspace(x_min, x_max, 1000)

# Default
n_seeds = 50
n_toys_per_seed = 100

# Generate seeds
default_seed = 42
np.random.seed(default_seed)
seeds = np.random.randint(0, 10000, size=n_seeds)

In [ ]:
# Set up models
toy_models = [
    emm.f1(x),
    emm.f2(x),
    emm.f3(x),
    emm.f4(x),
]

model_primitives = [
    emm.f1,
    emm.f2,
    emm.f3,
    emm.f4,
]
for k in [2, 3, 4]:
    model_primitives.append(
        emm.models.ModelPrimitive(
            emm.ExponentialMixtureModel,
            k,
            data_mean=700,
            name=f"ExponentialMixture-{k}",
        )
    )

print(f"Will be submitting {n_seeds * len(toy_models)} tasks, with {n_toys_per_seed} toys fitting {len(model_primitives)} models.")

# Fit toy models to data
for toy_model in toy_models:
    print(f"Fitting toy model: {toy_model.name}")
    toy_model.pdf.fitTo(data)

In [ ]:
# Run jobs
tasks = []
for seed in seeds:
    for toy_model in toy_models:
        tasks.append(so.Task(bias.run_bias_fits, x, toy_model, model_primitives, seed, n_toys_per_seed, n, x_grid))

# _ = so.run_tasks(
#     tasks,
#     use_condor=True,
#     condor_job_name="bias",
#     env_wrapper=so.run_in_mamba,
#     clear_logs=True
# )

In [ ]:
# Load and combine results
results = bias.get_bias_results(
    toy_models,
    seeds,
    n_toys_per_seed,
    n,
 )

In [ ]:
_ = bias.plot_bias(
    x,
    toy_models,
    results,
    x_grid,
    range=(500, 3500),
)